In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

SIZE = 10
DEGREE = 2
EPISODES = 10000
STEP_SIZE_0 = 1*(10**-2)
STEP_SIZE_1 = 1*(10**-4)
DISCOUNT = .6
SHOW_EPISODE = 100


def main():
    epsilon = 1
    average = 0
    step_size = STEP_SIZE_0
    weights = np.full((DEGREE+1,DEGREE+1,DEGREE+1,DEGREE+1,DEGREE+1,DEGREE+1), 0.0)

    for current in range(EPISODES):
        pos = (np.random.choice([0,1])*(SIZE-1),np.random.choice([0,1])*(SIZE-1))
        goal = (np.random.randint(3,SIZE-3),np.random.randint(3,SIZE-3))
        dist = np.subtract(goal,pos)
        count = 0

        while((dist[0]!=0)and(dist[1]!=0)):

            a = pick_action(pos,goal,epsilon,weights)
            action = (round(math.cos(a)),round(math.sin(a)))
            new_pos = find_new(action,pos)
            new_dist = np.subtract(goal,new_pos)

            features = np.empty((DEGREE+1,DEGREE+1,DEGREE+1,DEGREE+1,DEGREE+1,DEGREE+1))
            next_features = np.empty((DEGREE+1,DEGREE+1,DEGREE+1,DEGREE+1,DEGREE+1,DEGREE+1))
            reward = find_reward(new_pos,goal)
            for start_x in range(DEGREE+1):
                for start_y in range(DEGREE+1):
                    for end_x in range(DEGREE+1):
                        for end_y in range(DEGREE+1):
                            for x_dist in range(DEGREE+1):
                                for y_dist in range(DEGREE+1):
                                    features[start_x][start_y][end_x][end_y][x_dist][y_dist] = ((pos[0]/SIZE)**start_x)*((pos[1]/SIZE)**start_y)*((goal[0]/SIZE)**end_x)*((goal[1]/SIZE)**end_y)*((dist[0]/SIZE)**x_dist)*((dist[1]/SIZE)**y_dist)
                                    next_features[start_x][start_y][end_x][end_y][x_dist][y_dist] = ((new_pos[0]/SIZE)**start_x)*((new_pos[1]/SIZE)**start_y)*((goal[0]/SIZE)**end_x)*((goal[1]/SIZE)**end_y)*((new_dist[0]/SIZE)**x_dist)*((new_dist[1]/SIZE)**y_dist)

            old_value = np.sum(np.multiply(weights,features))
            new_value = np.sum(np.multiply(weights,next_features))

            weights = np.add(weights,step_size*(reward+DISCOUNT*new_value-old_value)*features)

            pos = new_pos
            dist = new_dist
            count += 1

        if((current+1)%SHOW_EPISODE==0):
            show_current(weights,goal,current,average)
            average = 0
        else:
            average = (average*(current%SHOW_EPISODE)+count)/((current+1)%SHOW_EPISODE)
        epsilon = epsilon - epsilon/(EPISODES//2-1)
        step_size = STEP_SIZE_0/(1+((STEP_SIZE_0-STEP_SIZE_1)/(STEP_SIZE_1*EPISODES))*current)

    #print_episode(weights,goal)


def show_current(weights,goal,current,average):
    print('Episode: '+str(current+1)+'\nAverage: '+str(average)+'\nGoal: '+str(goal),end='\n\n')
    ax = plt.axes(projection='3d')
    x,y = np.meshgrid(np.arange(SIZE),np.arange(SIZE))
    z = np.zeros((SIZE,SIZE))
    for i in range(SIZE):
        for j in range(SIZE):
            z[i][j] = find_value((i,j),goal,np.subtract(goal,(i,j)),weights)
    ax.plot_surface(x,y,z)
    plt.show()
    print_episode(weights,goal)

def pick_action(pos,goal,epsilon,weights):
    if(np.random.random()<epsilon):
        return(np.random.choice([0,np.pi/2,np.pi,3*np.pi/2]))
    else:
        return((np.pi/2)*arg_max(pos,goal,weights))

def arg_max(pos,goal,weights):
    values = np.empty(4)
    for i in range(4):
        new_pos = (pos[0]+round(math.cos(np.pi/2*i)),pos[1]+round(math.sin(np.pi/2*i)))
        if((new_pos[0]>=0) and (new_pos[0]<=SIZE-1) and (new_pos[1]>=0) and (new_pos[1]<=SIZE-1)):
            values[i] = find_value(new_pos,goal,np.subtract(goal,new_pos),weights)
        else:
            values[i] = -math.inf
    return(np.argmax(values))

def find_value(pos,goal,dist,weights):
    temp = 0
    for start_x in range(DEGREE+1):
        for start_y in range(DEGREE+1):
            for end_x in range(DEGREE+1):
                for end_y in range(DEGREE+1):
                    for x_dist in range(DEGREE+1):
                        for y_dist in range(DEGREE+1):
                            temp += (weights[start_x][start_y][end_x][end_y][x_dist][y_dist])*((pos[0]/SIZE)**start_x)*((pos[1]/SIZE)**start_y)*((goal[0]/SIZE)**end_x)*((goal[1]/SIZE)**end_y)*((dist[0]/SIZE)**x_dist)*((dist[1]/SIZE)**y_dist)
    return(temp)

def find_new(action,pos):
    temp1 = action[0]+pos[0]
    if(temp1>=SIZE):
        temp1 = SIZE-1
    elif(temp1<0):
        temp1 = 0
    temp2 = action[1]+pos[1]
    if(temp2>=SIZE):
        temp2 = SIZE-1
    elif(temp2<0):
        temp2 = 0
    return(temp1,temp2)

def find_reward(pos,goal):
    if(pos==goal):
        return 1
    else:
        return -.1

def print_episode(weights,goal):
    for y in range(SIZE):
        line = ""
        for x in range(SIZE):
            a = arg_max((x,y),goal,weights)
            if(a==0):
                line += "> "
            elif(a==1):
                line += "V "
            elif(a==2):
                line += "< "
            else:
                line += "^ "
        print(line)


main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import itertools
import math

SIZE = 10
DEGREE = 2
EPISODES = 10000
STEP_SIZE_0 = 1*(10**-1)
STEP_SIZE_1 = 5*(10**-5)
DISCOUNT = .5
SHOW_EPISODE = 100


def main():
    epsilon = 1
    average = 0
    step_size = STEP_SIZE_0
    weights = np.full((DEGREE+1,)*6, 0.0)

    for current in range(EPISODES):
        pos = (np.random.choice([0,1])*(SIZE-1),np.random.choice([0,1])*(SIZE-1))
        goal = (np.random.randint(3,SIZE-3),np.random.randint(3,SIZE-3))
        dist = np.subtract(goal,pos)
        count = 0

        while((dist[0]!=0)and(dist[1]!=0)):

            a = pick_action(pos,goal,epsilon,weights)
            action = (round(math.cos(a)),round(math.sin(a)))
            new_pos = find_new(action,pos)
            new_dist = np.subtract(goal,new_pos)

            features = calc_features(pos,goal,dist)
            next_features = calc_features(new_pos,goal,new_dist)
            reward = find_reward(new_pos,goal)

            old_value = np.sum(np.multiply(weights,features))
            new_value = np.sum(np.multiply(weights,next_features))

            weights = np.add(weights,step_size*(reward+DISCOUNT*new_value-old_value)*features)

            pos = new_pos
            dist = new_dist
            count += 1

        if((current+1)%SHOW_EPISODE==0):
            show_current(weights,goal,current,average)
            average = 0
        else:
            average = (average*(current%SHOW_EPISODE)+count)/((current+1)%SHOW_EPISODE)
        epsilon = epsilon - epsilon/(EPISODES//2-1)
        step_size = STEP_SIZE_0/(1+((STEP_SIZE_0-STEP_SIZE_1)/(STEP_SIZE_1*EPISODES))*current)


def show_current(weights,goal,current,average):
    print('Episode: '+str(current+1)+'\nAverage: '+str(average)+'\nGoal: '+str(goal),end='\n\n')
    ax = plt.axes(projection='3d')
    x,y = np.meshgrid(np.arange(SIZE),np.arange(SIZE))
    z = np.zeros((SIZE,SIZE))
    for i in range(SIZE):
        for j in range(SIZE):
            z[i][j] = find_value((i,j),goal,np.subtract(goal,(i,j)),weights)
    ax.plot_surface(x,y,z)
    plt.show()
    print_episode(weights,goal)

def calc_features(pos,goal,dist):
    indices = np.indices((DEGREE+1,)*6)
    start_x, start_y, end_x, end_y, x_dist, y_dist = [indices[i].flatten() for i in range(6)]
    temp = (
        ((pos[0]/SIZE)**start_x)*((pos[1]/SIZE)**start_y)*
        ((goal[0]/SIZE)**end_x)*((goal[1]/SIZE)**end_y)*
        ((dist[0]/SIZE)**x_dist)*((dist[1]/SIZE)**y_dist)
    )
    temp = temp.reshape((DEGREE+1,)*6)
    return(temp)

def pick_action(pos,goal,epsilon,weights):
    if(np.random.random()<epsilon):
        return(np.random.choice([0,np.pi/2,np.pi,3*np.pi/2]))
    else:
        return((np.pi/2)*arg_max(pos,goal,weights))

def arg_max(pos,goal,weights):
    values = np.zeros(4)
    for i in range(4):
        new_pos = (pos[0]+round(math.cos(np.pi/2*i)),pos[1]+round(math.sin(np.pi/2*i)))
        if((new_pos[0]>=0) and (new_pos[0]<=SIZE-1) and (new_pos[1]>=0) and (new_pos[1]<=SIZE-1)):
            values[i] = find_value(new_pos,goal,np.subtract(goal,new_pos),weights)
        else:
            values[i] = -math.inf
    return(np.argmax(values))

def find_value(pos,goal,dist,weights):
    flat_w = weights.flatten()
    indices = np.indices((DEGREE+1,)*6)
    start_x, start_y, end_x, end_y, x_dist, y_dist = [indices[i].flatten() for i in range(6)]
    values = flat_w*(
        ((pos[0]/SIZE)**start_x)*((pos[1]/SIZE)**start_y)*
        ((goal[0]/SIZE)**end_x)*((goal[1]/SIZE)**end_y)*
        ((dist[0]/SIZE)**x_dist)*((dist[1]/SIZE)**y_dist)
    )
    return values.sum()

def find_new(action,pos):
    temp1 = min(SIZE-1,max(0,action[0]+pos[0]))
    temp2 = min(SIZE-1,max(0,action[1]+pos[1]))
    return(temp1,temp2)

def find_reward(pos,goal):
    if(pos==goal):
        return 1
    else:
        return -.1

def print_episode(weights,goal):
    sym = {0: ">", 1: "V", 2: "<", 3: "^"}
    for y in range(SIZE):
        line = ' '.join(sym[arg_max((x,y),goal,weights)] for x in range(SIZE))
        print(line)
    print('\n')

main()

Output hidden; open in https://colab.research.google.com to view.